<a href="https://colab.research.google.com/github/Aham0803/PySpark/blob/main/Sales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [72]:
%pip install pyspark

In [73]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [74]:
spark = SparkSession.builder\
.appName("RetailSalesProject")\
.getOrCreate()
print("Spark version" , spark.version)
print("Spark session  created")

Spark version 4.0.4
Spark session  created


In [75]:
data=[
(1001,"Rahul","Laptop",1,50000,"Bangalore","2026-10-01"),
(1002,"Priya","Mouse",2,500,"Mumbai","2026-10-01"),
(1003,"Amit","Keyboard",1,1500,"Delhi","2026-10-02"),
(1004,"Rahul","Laptop",1,50000,"Bangalore","2026-10-02"),
(1005,"Neha","Monitor",None,12000,"Chennai","2026-10-03"),
(1006,"Arjun","Mouse",3,500,"Mumbai","2026-10-03"),
(1007,"Priya","Keyboard",2,1500,"Mumbai","2026-10-04"),
(1008,"Sneha","Laptop",1,50000,"Delhi","2026-10-04"),
(1009,"Vikram","Monitor",2,12000,"Bangalore","2026-10-05"),
(1010,"Neha","Mouse",4,500,"Chennai","2026-10-05"),
(1011,"Karan","Keyboard",1,1500,"Delhi","2026-10-06"),
(1012,"Amit","Laptop",2,50000,"Bangalore","2026-10-06"),
(1013,"Rahul","Mouse",2,500,"Bangalore","2026-10-07"),
(1014,"Sneha","Monitor",1,12000,"Mumbai","2026-10-07"),
(1015,"Arjun","Laptop",1,50000,"Delhi","2026-10-08"),
(1015,"Arjun","Laptop",1,50000,"Delhi","2026-10-08")
]


columns=[
"order_id","customer", "product",
 "qty", "price",
"city",
"order_date"
]

In [76]:
df = spark.createDataFrame(data,columns)
df.show()

+--------+--------+--------+----+-----+---------+----------+
|order_id|customer| product| qty|price|     city|order_date|
+--------+--------+--------+----+-----+---------+----------+
|    1001|   Rahul|  Laptop|   1|50000|Bangalore|2026-10-01|
|    1002|   Priya|   Mouse|   2|  500|   Mumbai|2026-10-01|
|    1003|    Amit|Keyboard|   1| 1500|    Delhi|2026-10-02|
|    1004|   Rahul|  Laptop|   1|50000|Bangalore|2026-10-02|
|    1005|    Neha| Monitor|NULL|12000|  Chennai|2026-10-03|
|    1006|   Arjun|   Mouse|   3|  500|   Mumbai|2026-10-03|
|    1007|   Priya|Keyboard|   2| 1500|   Mumbai|2026-10-04|
|    1008|   Sneha|  Laptop|   1|50000|    Delhi|2026-10-04|
|    1009|  Vikram| Monitor|   2|12000|Bangalore|2026-10-05|
|    1010|    Neha|   Mouse|   4|  500|  Chennai|2026-10-05|
|    1011|   Karan|Keyboard|   1| 1500|    Delhi|2026-10-06|
|    1012|    Amit|  Laptop|   2|50000|Bangalore|2026-10-06|
|    1013|   Rahul|   Mouse|   2|  500|Bangalore|2026-10-07|
|    1014|   Sneha| Moni

In [77]:
df.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- customer: string (nullable = true)
 |-- product: string (nullable = true)
 |-- qty: long (nullable = true)
 |-- price: long (nullable = true)
 |-- city: string (nullable = true)
 |-- order_date: string (nullable = true)



In [78]:
print("Total records: " , df.count())

Total records:  16


In [79]:
df.select([
sum(col(c).isNull().cast("int")).alias(c)
for c in columns
]).show()

+--------+--------+-------+---+-----+----+----------+
|order_id|customer|product|qty|price|city|order_date|
+--------+--------+-------+---+-----+----+----------+
|       0|       0|      0|  1|    0|   0|         0|
+--------+--------+-------+---+-----+----+----------+



In [80]:
df.select("order_id" , "customer" ,"product" ,"qty").show()

+--------+--------+--------+----+
|order_id|customer| product| qty|
+--------+--------+--------+----+
|    1001|   Rahul|  Laptop|   1|
|    1002|   Priya|   Mouse|   2|
|    1003|    Amit|Keyboard|   1|
|    1004|   Rahul|  Laptop|   1|
|    1005|    Neha| Monitor|NULL|
|    1006|   Arjun|   Mouse|   3|
|    1007|   Priya|Keyboard|   2|
|    1008|   Sneha|  Laptop|   1|
|    1009|  Vikram| Monitor|   2|
|    1010|    Neha|   Mouse|   4|
|    1011|   Karan|Keyboard|   1|
|    1012|    Amit|  Laptop|   2|
|    1013|   Rahul|   Mouse|   2|
|    1014|   Sneha| Monitor|   1|
|    1015|   Arjun|  Laptop|   1|
|    1015|   Arjun|  Laptop|   1|
+--------+--------+--------+----+



In [83]:
df.groupBy("order_id").count().filter(col("count")>1
).show()


+--------+-----+
|order_id|count|
+--------+-----+
|    1015|    2|
+--------+-----+



In [84]:
df_clean=df.dropDuplicates(["order_id"])
print("Records before dup cleaning ",df.count())
print("Records after dup cleaning ",df_clean.count())

Records before dup cleaning  16
Records after dup cleaning  15


In [85]:
df_clean.filter(col("qty").isNull()).show()

+--------+--------+-------+----+-----+-------+----------+
|order_id|customer|product| qty|price|   city|order_date|
+--------+--------+-------+----+-----+-------+----------+
|    1005|    Neha|Monitor|NULL|12000|Chennai|2026-10-03|
+--------+--------+-------+----+-----+-------+----------+



In [86]:
df_clean = df_clean.dropna(
    subset = ["qty"]
)
print("records after removing missing qty " , df_clean.count())

records after removing missing qty  14


In [87]:
df_clean.filter(
    col("qty").isNull()
).show()

+--------+--------+-------+---+-----+----+----------+
|order_id|customer|product|qty|price|city|order_date|
+--------+--------+-------+---+-----+----+----------+
+--------+--------+-------+---+-----+----+----------+



In [88]:
df_clean.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- customer: string (nullable = true)
 |-- product: string (nullable = true)
 |-- qty: long (nullable = true)
 |-- price: long (nullable = true)
 |-- city: string (nullable = true)
 |-- order_date: string (nullable = true)



In [89]:
df_clean = df_clean.withColumn(
    "order_date",
    to_date(col("order_date"), "yyyy-MM--dd")
)
df_clean.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- customer: string (nullable = true)
 |-- product: string (nullable = true)
 |-- qty: long (nullable = true)
 |-- price: long (nullable = true)
 |-- city: string (nullable = true)
 |-- order_date: date (nullable = true)



In [91]:
df_clean = df_clean.withColumn(
    "qty",
    col("qty").cast("int")
)
df_clean = df_clean.withColumn(
    "price",
    col("price").cast("double")
)
df_clean.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- customer: string (nullable = true)
 |-- product: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- city: string (nullable = true)
 |-- order_date: date (nullable = true)

